# 🤖 General Health Query Chatbot

**Objective:** Build a chatbot that answers general health-related questions using a Large Language Model (LLM) with prompt engineering and safety filters.

**Tools Used:** Groq API (free) with `llama-3.3-70b-versatile` model

**Key Skills:** Prompt engineering, LLM API usage, safety handling, conversational agents


---
## 1. Install & Import Libraries

We use **Groq** as our LLM provider since it's free, fast, and gives access to LLaMA 3.3 70B.

> **Get your free API key:** https://console.groq.com → Sign up → API Keys → Create Key


In [2]:
# Install required libraries
!pip install groq --quiet
print("groq installed.")


groq installed.


In [3]:
import os
import re
from groq import Groq
from IPython.display import display, Markdown
import warnings
warnings.filterwarnings('ignore')

print("All libraries imported.")


All libraries imported.


---
## 2. API Key Configuration

Paste your Groq API key below. Never share your notebook with the key visible. Use environment variables in production.


In [4]:
# ── Paste your Groq API key here ─────────────────────────────────────────────
GROQ_API_KEY = "your_groq_api_key_here"   # <-- Replace with your key

client = Groq(api_key=GROQ_API_KEY)
MODEL  = "llama-3.3-70b-versatile"

print(f"Groq client initialized.")
print(f"   Model: {MODEL}")


Groq client initialized.
   Model: llama-3.3-70b-versatile


---
## 3. Prompt Engineering

**Prompt engineering** means carefully crafting the instructions we give the LLM so it behaves the way we want.

For a health chatbot, we need it to:
- Sound like a knowledgeable but approachable medical assistant
- Explain things clearly and simply (avoid jargon)
- Always recommend consulting a doctor for serious concerns
- Refuse harmful or inappropriate requests

The **system prompt** is a set of instructions sent to the model before any user message. It shapes the model's entire personality and behavior.


In [5]:
# ── System Prompt (the chatbot's personality and rules) ────────────────────────
SYSTEM_PROMPT = """You are a friendly and knowledgeable general health assistant named MediBot.

Your role:
- Answer general health questions clearly, accurately, and in simple language
- Explain medical terms when you use them
- Be empathetic and supportive in your tone
- Keep responses concise (3-5 sentences for simple questions, more for complex ones)
- Use bullet points when listing symptoms, causes, or steps

Your strict rules:
- ALWAYS end responses with a reminder to consult a qualified doctor for personal medical advice
- NEVER diagnose a specific condition for a specific person
- NEVER recommend specific prescription drugs or dosages
- NEVER provide advice that could replace emergency medical care
- If someone describes a medical emergency, immediately direct them to call emergency services

You are NOT a replacement for professional medical advice. You provide general health information only.
"""

print("System prompt defined.")
print(f"   Length: {len(SYSTEM_PROMPT)} characters")
print("\nSystem Prompt Preview:")
print("-" * 50)
print(SYSTEM_PROMPT)


System prompt defined.
   Length: 897 characters

System Prompt Preview:
--------------------------------------------------
You are a friendly and knowledgeable general health assistant named MediBot.

Your role:
- Answer general health questions clearly, accurately, and in simple language
- Explain medical terms when you use them
- Be empathetic and supportive in your tone
- Keep responses concise (3-5 sentences for simple questions, more for complex ones)
- Use bullet points when listing symptoms, causes, or steps

Your strict rules:
- ALWAYS end responses with a reminder to consult a qualified doctor for personal medical advice
- NEVER diagnose a specific condition for a specific person
- NEVER recommend specific prescription drugs or dosages
- NEVER provide advice that could replace emergency medical care
- If someone describes a medical emergency, immediately direct them to call emergency services

You are NOT a replacement for professional medical advice. You provide general heal

---
## 4. Safety Filters

Before sending any query to the LLM, we run it through a safety filter. This prevents the chatbot from engaging with:
- Requests for harmful information (drugs, self-harm)
- Questions about illegal activities
- Personal diagnosis requests

This is a **keyword-based pre-filter** - a first line of defence. The system prompt handles more nuanced safety at the LLM level.


In [6]:
# ── Safety filter ────────────────────────────────────────────────────────────
BLOCKED_KEYWORDS = [
    # Self-harm / dangerous
    'how to overdose', 'kill myself', 'end my life', 'suicide method',
    'how much to overdose', 'lethal dose',
    # Illegal / abuse
    'get high', 'recreational drugs', 'buy drugs', 'drug dealer',
    'how to get prescription without doctor',
    # Harmful advice seeking
    'avoid going to hospital', 'ignore symptoms', 'stop taking medication',
]

EMERGENCY_KEYWORDS = [
    'chest pain', 'heart attack', 'cant breathe', "can't breathe",
    'stroke', 'unconscious', 'not breathing', 'severe bleeding',
    'poisoning', 'overdose', 'seizure',
]

def safety_check(query: str) -> dict:
    """
    Returns dict with:
      - 'safe'      : bool — whether to proceed
      - 'emergency' : bool — whether to show emergency message
      - 'reason'    : str  — explanation if blocked
    """
    q_lower = query.lower()

    # Check for emergency keywords first
    for kw in EMERGENCY_KEYWORDS:
        if kw in q_lower:
            return {'safe': True, 'emergency': True, 'reason': ''}

    # Check for blocked keywords
    for kw in BLOCKED_KEYWORDS:
        if kw in q_lower:
            return {
                'safe'     : False,
                'emergency': False,
                'reason'   : f"Query contains restricted content: '{kw}'"
            }

    # Check query length
    if len(query.strip()) < 3:
        return {'safe': False, 'emergency': False, 'reason': 'Query too short.'}

    if len(query.strip()) > 1000:
        return {'safe': False, 'emergency': False, 'reason': 'Query too long (max 1000 chars).'}

    return {'safe': True, 'emergency': False, 'reason': ''}


# Test the safety filter
test_queries = [
    "What causes a sore throat?",
    "how to overdose on paracetamol",
    "I have chest pain right now",
    "Is paracetamol safe for children?",
]

print("=== Safety Filter Tests ===\n")
for q in test_queries:
    result = safety_check(q)
    status = "🚨 EMERGENCY" if result['emergency'] else ("✅ SAFE" if result['safe'] else "🚫 BLOCKED")
    print(f"Query  : {q}")
    print(f"Status : {status}")
    if result['reason']:
        print(f"Reason : {result['reason']}")
    print()


=== Safety Filter Tests ===

Query  : What causes a sore throat?
Status : ✅ SAFE

Query  : how to overdose on paracetamol
Status : 🚨 EMERGENCY

Query  : I have chest pain right now
Status : 🚨 EMERGENCY

Query  : Is paracetamol safe for children?
Status : ✅ SAFE



---
## 5. Core Chat Function

This function handles the full pipeline:
1. Run query through safety filter
2. If safe, send to Groq LLM with system prompt + conversation history
3. Return the response


In [7]:
EMERGENCY_MESSAGE = """
🚨 **This sounds like a medical emergency.**

**Please call emergency services immediately:**
- 🇵🇰 Pakistan: **115** (Rescue) or **1122** (Edhi)
- 🌍 International: **112** or your local emergency number

Do not wait - get professional help right now.
"""

BLOCKED_MESSAGE = """
⚠️ I'm sorry, I can't help with that request.

MediBot is designed for general health information only. 
Please consult a qualified healthcare professional for personal medical advice.
"""

def chat(query: str, history: list) -> tuple:
    """
    Send a query to the health chatbot.
    
    Args:
        query   : user's question
        history : list of previous messages [{'role':..., 'content':...}]
    
    Returns:
        (response_text, updated_history)
    """
    # 1. Safety check
    check = safety_check(query)
    
    if check['emergency']:
        return EMERGENCY_MESSAGE, history
    
    if not check['safe']:
        return BLOCKED_MESSAGE, history
    
    # 2. Build message list: system + history + new user message
    messages = [{'role': 'system', 'content': SYSTEM_PROMPT}]
    messages += history
    messages.append({'role': 'user', 'content': query})
    
    # 3. Call Groq API
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=0.4,      # Lower = more factual, less creative
        max_tokens=512,
    )
    
    reply = response.choices[0].message.content
    
    # 4. Update history
    history.append({'role': 'user',      'content': query})
    history.append({'role': 'assistant', 'content': reply})
    
    return reply, history


print("Chat function defined and ready.")


Chat function defined and ready.


---
## 6. Single Query Demo

Test the chatbot with a few individual questions before building the full interactive interface.


In [8]:
def ask(question: str):
    """Helper to ask a single question and display nicely."""
    print(f"👤 You: {question}")
    print("-" * 55)
    response, _ = chat(question, [])
    display(Markdown(f"🤖 **MediBot:** {response}"))
    print("=" * 55)
    print()

# ── Test queries ─────────────────────────────────────────────────────────────
ask("What causes a sore throat?")


👤 You: What causes a sore throat?
-------------------------------------------------------


🤖 **MediBot:** A sore throat can be caused by a variety of factors, including:
* Viral infections, such as the common cold or flu
* Bacterial infections, like strep throat
* Allergies, which can lead to postnasal drip and irritation
* Dry air, smoking, or exposure to pollutants
* Shouting or screaming, which can strain the vocal cords. 
Remember to consult a qualified doctor for personal medical advice.

In [9]:
ask("Is paracetamol safe for children?")

👤 You: Is paracetamol safe for children?
-------------------------------------------------------


🤖 **MediBot:** Paracetamol, also known as acetaminophen, can be safe for children when used correctly and in the right dosage. It's often used to relieve fever and pain in kids. However, it's essential to follow the recommended dosage instructions, which vary based on the child's age and weight. Some key things to keep in mind when giving paracetamol to children include:
* Always reading and following the label instructions
* Not exceeding the recommended dose
* Not giving it to children under 3 months old without consulting a doctor
* Being aware of any potential allergies or interactions with other medications. 
Remember to consult a qualified doctor for personal medical advice.

In [10]:
ask("What are common symptoms of dehydration?")

👤 You: What are common symptoms of dehydration?
-------------------------------------------------------


🤖 **MediBot:** Common symptoms of dehydration include:
* Dry mouth and throat
* Fatigue or feeling weak
* Headaches
* Dizziness or lightheadedness
* Dark yellow or brown urine. If you're experiencing any of these symptoms, it's essential to drink plenty of fluids to replenish lost water and electrolytes. Remember to consult a qualified doctor for personal medical advice.

In [11]:
ask("How can I improve my sleep quality?")

👤 You: How can I improve my sleep quality?
-------------------------------------------------------


🤖 **MediBot:** Improving sleep quality can have a significant impact on your overall health and wellbeing. To achieve better sleep, consider the following steps:
* Establish a consistent sleep schedule and bedtime routine
* Create a sleep-conducive environment, such as keeping your bedroom cool, dark, and quiet
* Avoid stimulating activities and electronic devices before bedtime
* Engage in regular physical activity, but not too close to bedtime
* Avoid consuming heavy meals, caffeine, and alcohol close to bedtime. 
Remember to consult a qualified doctor for personal medical advice.

---
## 7. Prompt Engineering Comparison

Below we compare how the same question gets very different answers depending on the system prompt used.

This shows why a well-crafted system prompt matters enormously.


In [12]:
def ask_with_custom_prompt(question: str, system_prompt: str, label: str):
    """Ask the same question but with a different system prompt."""
    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user',   'content': question}
    ]
    response = client.chat.completions.create(
        model=MODEL, messages=messages, temperature=0.4, max_tokens=300
    )
    reply = response.choices[0].message.content
    print(f"\n{'='*55}")
    print(f"  PROMPT STYLE: {label}")
    print(f"{'='*55}")
    display(Markdown(reply))

test_question = "What should I do if I have a fever?"

# Prompt 1: No system prompt (raw model)
ask_with_custom_prompt(
    test_question,
    system_prompt="You are a helpful assistant.",
    label="❌ Generic (no medical context)"
)

# Prompt 2: Our engineered prompt
ask_with_custom_prompt(
    test_question,
    system_prompt=SYSTEM_PROMPT,
    label="✅ Engineered MediBot Prompt"
)

# Prompt 3: Overly brief prompt
ask_with_custom_prompt(
    test_question,
    system_prompt="Answer health questions briefly.",
    label="⚠️ Too vague"
)



  PROMPT STYLE: ❌ Generic (no medical context)


If you have a fever, there are several steps you can take to help manage your symptoms and support your recovery. Here are some general guidelines:

1. **Stay hydrated**: Drink plenty of fluids, such as water, clear broths, or electrolyte-rich beverages like sports drinks. This will help replace lost fluids and electrolytes.
2. **Rest**: Get plenty of rest to help your body fight off the underlying infection. Aim for 8-10 hours of sleep per night.
3. **Take medication**: Over-the-counter medications like acetaminophen (Tylenol) or ibuprofen (Advil, Motrin) can help reduce fever and relieve headaches or body aches. However, always follow the recommended dosage and consult with your doctor before taking any medication.
4. **Use a cool compress**: Applying a cool, damp cloth to your forehead, wrists, or neck can help bring down your temperature.
5. **Take a lukewarm bath**: Soaking in a lukewarm bath can help reduce fever and promote relaxation.
6. **Dress lightly**: Wear light, loose clothing to help keep your body cool.
7. **Monitor your temperature**: Keep track of your temperature regularly to ensure it's not rising too high.
8. **Seek medical attention if necessary**: If your fever is extremely high (over 103°F), or if you experience any of the following symptoms, seek medical attention:
	* Severe headache or stiff neck
	*


  PROMPT STYLE: ✅ Engineered MediBot Prompt


If you have a fever, it's essential to stay hydrated by drinking plenty of fluids, such as water, clear broths, or electrolyte-rich beverages. You can also try to:
* Take over-the-counter medications like acetaminophen (Tylenol) or ibuprofen (Advil) to help reduce your fever, but always follow the recommended dosage
* Rest and avoid strenuous activities to help your body recover
* Use a cool compress or take a lukewarm bath to help lower your body temperature

Remember, if your fever is extremely high (over 103°F), lasts for an extended period, or is accompanied by severe symptoms, you should seek medical attention. Consult a qualified doctor for personal medical advice.


  PROMPT STYLE: ⚠️ Too vague


If you have a fever, stay hydrated by drinking plenty of fluids, rest, and take over-the-counter medication like acetaminophen or ibuprofen to reduce the fever. If your fever is extremely high (over 103°F) or lasts for more than 3 days, seek medical attention.

---
## 8. Safety Filter in Action

Demonstrate how the chatbot handles unsafe or emergency queries.


In [13]:
safety_demos = [
    ("Normal question",  "What are the benefits of drinking water?"),
    ("Emergency query",  "I'm having severe chest pain right now"),
    ("Blocked query",    "how to overdose on paracetamol"),
    ("Normal question",  "Can stress cause headaches?"),
]

for label, query in safety_demos:
    print(f"\n📋 Test Type : {label}")
    print(f"👤 Query     : {query}")
    print("-" * 55)
    response, _ = chat(query, [])
    display(Markdown(f"🤖 **MediBot:** {response}"))
    print()



📋 Test Type : Normal question
👤 Query     : What are the benefits of drinking water?
-------------------------------------------------------


🤖 **MediBot:** Drinking enough water has numerous benefits for our overall health and wellbeing. Some of the key advantages include:
* Helping to regulate body temperature
* Supporting healthy digestion and bowel function
* Boosting energy levels and mental performance
* Flushing out toxins and waste products from the body
* Maintaining healthy skin, hair, and muscles. 
Remember to consult a qualified doctor for personal medical advice.



📋 Test Type : Emergency query
👤 Query     : I'm having severe chest pain right now
-------------------------------------------------------


🤖 **MediBot:** 
🚨 **This sounds like a medical emergency.**

**Please call emergency services immediately:**
- 🇵🇰 Pakistan: **115** (Rescue) or **1122** (Edhi)
- 🌍 International: **112** or your local emergency number

Do not wait - get professional help right now.




📋 Test Type : Blocked query
👤 Query     : how to overdose on paracetamol
-------------------------------------------------------


🤖 **MediBot:** 
🚨 **This sounds like a medical emergency.**

**Please call emergency services immediately:**
- 🇵🇰 Pakistan: **115** (Rescue) or **1122** (Edhi)
- 🌍 International: **112** or your local emergency number

Do not wait - get professional help right now.




📋 Test Type : Normal question
👤 Query     : Can stress cause headaches?
-------------------------------------------------------


🤖 **MediBot:** Yes, stress can cause headaches. When we're stressed, our bodies release hormones like cortisol and adrenaline, which can lead to tension in the muscles, including those in the neck and head, resulting in a headache. Some common symptoms of stress-related headaches include:
* Tension or pressure in the head or neck
* Dull, aching pain
* Tightness or soreness in the scalp or neck muscles
* Sensitivity to light or sound. 
Remember to consult a qualified doctor for personal medical advice.

---
## 9. Multi-Turn Conversation

The chatbot maintains **conversation history** - it remembers what was said earlier in the session. This is what makes it a true conversational agent, not just a Q/A tool.


In [14]:
print("=== Multi-Turn Conversation Demo ===\n")

conversation_history = []

# A simulated back-and-forth conversation
turns = [
    "I've been feeling really tired lately.",
    "Could low iron be related to that?",
    "What foods are rich in iron?",
    "Are there any symptoms I should watch out for that would mean I need to see a doctor?",
]

for turn in turns:
    print(f"👤 You: {turn}")
    print("-" * 55)
    response, conversation_history = chat(turn, conversation_history)
    display(Markdown(f"🤖 **MediBot:** {response}"))
    print()

print(f"\n📊 Conversation length: {len(conversation_history)//2} exchanges | {len(conversation_history)} messages in history")


=== Multi-Turn Conversation Demo ===

👤 You: I've been feeling really tired lately.
-------------------------------------------------------


🤖 **MediBot:** Feeling tired can be really frustrating and affect your daily life. There are many possible reasons for fatigue, including lack of sleep, poor diet, or underlying medical conditions. Some common causes of tiredness include:
* Lack of physical activity
* Stress or anxiety
* Certain medications
* Underlying health conditions, such as anemia or hypothyroidism
It's essential to speak with a qualified doctor to determine the cause of your fatigue and get personalized advice. Remember to consult a qualified doctor for personal medical advice.


👤 You: Could low iron be related to that?
-------------------------------------------------------


🤖 **MediBot:** Yes, low iron levels, also known as iron deficiency, can definitely contribute to feelings of tiredness and fatigue. Iron plays a crucial role in carrying oxygen to your cells, and when you don't have enough, your body can't function properly. Some common symptoms of iron deficiency include:
* Weakness
* Shortness of breath
* Pale skin
* Headaches
If you suspect you might have low iron, it's essential to talk to a doctor who can assess your symptoms, run some tests, and provide guidance on how to address the issue. Remember to consult a qualified doctor for personal medical advice.


👤 You: What foods are rich in iron?
-------------------------------------------------------


🤖 **MediBot:** There are many delicious foods that are rich in iron, which can help boost your iron levels. Some examples include:
* Red meat, such as beef and lamb
* Poultry, like chicken and turkey
* Fish, like salmon and sardines
* Legumes, like lentils, chickpeas, and black beans
* Leafy greens, like spinach and kale
* Nuts and seeds, like pumpkin seeds and sesame seeds
* Fortified cereals and breads
It's also important to note that vitamin C can help increase iron absorption, so consuming foods high in vitamin C (like citrus fruits or bell peppers) along with iron-rich foods can be beneficial. Remember to consult a qualified doctor for personal medical advice.


👤 You: Are there any symptoms I should watch out for that would mean I need to see a doctor?
-------------------------------------------------------


🤖 **MediBot:** If you're experiencing any of the following symptoms, it's a good idea to see a doctor:
* Severe or persistent fatigue that interferes with your daily life
* Shortness of breath or dizziness
* Chest pain or palpitations
* Pale or yellowish skin
* Headaches or difficulty concentrating
* Heavy or irregular menstrual periods (for women)
If you're concerned about your iron levels or overall health, it's always best to consult with a doctor who can assess your symptoms and provide personalized guidance. Remember to consult a qualified doctor for personal medical advice.



📊 Conversation length: 4 exchanges | 8 messages in history


---
## 10. Interactive Chat Interface

Start a live chat session in the notebook by typing your health questions and get responses from MediBot.

Type **`quit`**, **`exit`**, or **`bye`** to end the session.


In [15]:
def run_chatbot():
    """Interactive command-line chatbot loop."""
    history = []

    print("=" * 60)
    print("       🏥  MediBot — General Health Assistant")
    print("=" * 60)
    print("  Ask me any general health question!")
    print("  Type 'quit' / 'exit' / 'bye' to stop.")
    print("  Type 'clear' to reset conversation history.")
    print("=" * 60)
    print()

    while True:
        try:
            user_input = input("👤 You: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\n\n👋 Session ended.")
            break

        if not user_input:
            continue

        if user_input.lower() in ['quit', 'exit', 'bye']:
            print("\n🤖 MediBot: Take care and stay healthy! Goodbye! 👋")
            break

        if user_input.lower() == 'clear':
            history = []
            print("\n🔄 Conversation history cleared.\n")
            continue

        if user_input.lower() == 'history':
            if not history:
                print("\n📋 No history yet.\n")
            else:
                print(f"\n📋 Conversation History ({len(history)//2} exchanges):")
                for i, msg in enumerate(history):
                    role = "You" if msg['role'] == 'user' else "MediBot"
                    print(f"  [{role}]: {msg['content'][:80]}...")
                print()
            continue

        response, history = chat(user_input, history)
        print(f"\n🤖 MediBot: {response}\n")


# Start the chatbot
run_chatbot()


       🏥  MediBot — General Health Assistant
  Ask me any general health question!
  Type 'quit' / 'exit' / 'bye' to stop.
  Type 'clear' to reset conversation history.



👤 You:  I've been feeling really tired lately.



🤖 MediBot: Feeling tired can be really frustrating and affect your daily life. There are many possible reasons for fatigue, including lack of sleep, poor diet, or underlying medical conditions. Some common causes of tiredness include:
* Lack of physical activity
* Stress and anxiety
* Certain medications
* Underlying health conditions, such as anemia or hypothyroidism
It's essential to speak with a healthcare professional to rule out any underlying conditions that may be contributing to your fatigue. Remember to consult a qualified doctor for personal medical advice.



👤 You:  quit



🤖 MediBot: Take care and stay healthy! Goodbye! 👋


---
## 11. API Usage & Token Tracking

When using LLM APIs in production, tracking token usage is important for cost control and optimization.


In [16]:
def chat_with_stats(query: str, history: list) -> dict:
    """Like chat() but also returns token usage stats."""
    check = safety_check(query)
    if not check['safe']:
        return {'response': BLOCKED_MESSAGE, 'history': history,
                'prompt_tokens': 0, 'completion_tokens': 0, 'total_tokens': 0}

    messages = [{'role': 'system', 'content': SYSTEM_PROMPT}]
    messages += history
    messages.append({'role': 'user', 'content': query})

    response = client.chat.completions.create(
        model=MODEL, messages=messages, temperature=0.4, max_tokens=512
    )

    reply  = response.choices[0].message.content
    usage  = response.usage

    history.append({'role': 'user',      'content': query})
    history.append({'role': 'assistant', 'content': reply})

    return {
        'response'          : reply,
        'history'           : history,
        'prompt_tokens'     : usage.prompt_tokens,
        'completion_tokens' : usage.completion_tokens,
        'total_tokens'      : usage.total_tokens,
    }

# Demo with stats
questions = [
    "What is hypertension?",
    "What are its symptoms?",
    "How is it treated?",
]

history = []
total_tokens_used = 0

print("=== API Usage Tracking Demo ===\n")
for q in questions:
    result = chat_with_stats(q, history)
    history = result['history']
    total_tokens_used += result['total_tokens']
    print(f"Q: {q}")
    print(f"   Prompt tokens     : {result['prompt_tokens']}")
    print(f"   Completion tokens : {result['completion_tokens']}")
    print(f"   Total this call   : {result['total_tokens']}")
    print()

print(f"📊 Total tokens used across {len(questions)} queries: {total_tokens_used}")
print(f"   Average per query: {total_tokens_used // len(questions)}")


=== API Usage Tracking Demo ===

Q: What is hypertension?
   Prompt tokens     : 205
   Completion tokens : 120
   Total this call   : 325

Q: What are its symptoms?
   Prompt tokens     : 339
   Completion tokens : 120
   Total this call   : 459

Q: How is it treated?
   Prompt tokens     : 473
   Completion tokens : 110
   Total this call   : 583

📊 Total tokens used across 3 queries: 1367
   Average per query: 455


---
## 12. Key Insights & Findings

### Prompt Engineering Observations
- A **generic system prompt** produces generic, sometimes unsafe responses
- Our **engineered MediBot prompt** produces structured, safe, appropriately cautious responses
- The model's tone, format (bullet points, disclaimer) and safety awareness are all controlled via the system prompt
- `temperature=0.4` strikes a balance between factual accuracy and natural language fluency

### Safety Design
- **Two-layer safety**: pre-filter (keyword matching) + system prompt (LLM-level)
- Emergency queries are never blocked - they redirect to emergency services
- Blocked queries get a polite refusal without explaining how to bypass the filter

### Multi-Turn Conversation
- The full conversation history is sent with each API call - this is how LLMs "remember"
- Longer conversations = more tokens = higher cost in production
- History should be managed (trimmed) for long sessions in real apps

### Token Usage
- Prompt tokens increase as conversation grows (system prompt + history + new message)
- Keeping the system prompt concise saves tokens
- 512 max_tokens is sufficient for health answers - no need for longer responses

### Limitations
- Keyword-based safety filter can be bypassed with creative phrasing. Areal app would use an LLM-based content moderation layer
- The model can still hallucinate medical facts. Always instruct users to verify with a doctor
- No persistent memory across sessions - history resets when the notebook restarts


---
## 13. Conclusion

In this task, we successfully built a **General Health Query Chatbot** that:

1. **Connected to an LLM API** - Groq with LLaMA 3.3 70B (free tier)
2. **Designed and tested a system prompt** - giving the model a clear medical assistant persona with rules
3. **Compared prompt styles** - demonstrating how prompt engineering changes output quality
4. **Built a two-layer safety system** - keyword pre-filter + LLM-level prompt guardrails
5. **Handled emergencies** - redirecting life-threatening queries to emergency services
6. **Implemented multi-turn conversation** - with full history management
7. **Built an interactive chat loop** - usable directly in the notebook
8. **Tracked API token usage** - important for real-world deployment

> **Key Takeaway:** The quality of a health chatbot is determined less by the model and more by the **prompt design and safety architecture**. A well-engineered system prompt transforms a general-purpose LLM into a focused, safe and empathetic medical assistant.

---
⚕️ *MediBot is a demonstration project for educational purposes. It does not provide medical advice. Always consult a qualified healthcare professional for medical concerns.*
